# KV Cache: Accelerating Autoregressive Generation

During autoregressive text generation, a language model produces one token at a time. At each step,
the attention mechanism needs key (K) and value (V) projections for **every previous token** in the
sequence. Without caching, this means redundantly recomputing K and V for all past tokens on every
single generation step -- an O(n^2) cost that grows quadratically with sequence length.

The **KV cache** solves this by storing previously computed K and V tensors and reusing them. Each
new generation step only computes K and V for the single new token, then appends them to the cache.
This reduces per-step attention from O(n) recomputation to O(1) new computation (plus an O(n)
lookup against the cache), yielding significant wall-clock speedups.

In this notebook we will:
1. Understand **why** KV caching is necessary
2. Build intuition with a simple list-based cache
3. Use a pre-allocated tensor-based cache for realistic workloads
4. Benchmark speed and analyze memory tradeoffs

In [ ]:
%matplotlib inline

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import time

from kv_cache import (
    KVCache,
    attention_with_cache,
    benchmark_with_without_cache,
    plot_cache_benchmarks,
    visualize_cache_memory,
    analyze_cache_efficiency,
)
from cache import KVCache as SimpleKVCache

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")
print(f"Device: cpu")

---
## 1. The Problem: Redundant Recomputation

In standard (non-cached) autoregressive generation, producing token `t` requires computing
attention over all `t` tokens. For the **next** token `t+1`, we recompute attention over all
`t+1` tokens from scratch -- even though the K and V for tokens `1..t` have not changed.

Let's measure how the cost scales with sequence length when we recompute everything each step.

In [ ]:
# Simulate the cost of full recomputation at each generation step
batch_size, num_heads, head_dim = 1, 4, 16
seq_lengths = [8, 16, 32, 64]

print("Full recomputation cost per generation step:")
print(f"{'Seq Len':>8} | {'Matmuls (Q*K^T)':>16} | {'Time (ms)':>10}")
print("-" * 42)

for seq_len in seq_lengths:
    # At step t, we compute attention over all t tokens
    total_ops = 0
    start = time.time()
    for t in range(1, seq_len + 1):
        q = torch.randn(batch_size, num_heads, t, head_dim)
        k = torch.randn(batch_size, num_heads, t, head_dim)
        v = torch.randn(batch_size, num_heads, t, head_dim)
        scores = torch.matmul(q, k.transpose(-2, -1)) / (head_dim ** 0.5)
        weights = torch.softmax(scores, dim=-1)
        _ = torch.matmul(weights, v)
        total_ops += t * t  # Each step does t x t attention matrix
    elapsed = (time.time() - start) * 1000
    print(f"{seq_len:>8} | {total_ops:>16,} | {elapsed:>9.2f}ms")

print("\nNotice: cost grows roughly as O(n^3) -- sum of t^2 for t=1..n.")

The numbers above show that full recomputation is extremely wasteful. For a sequence of length
64, we perform over 85,000 element-wise multiplications in the attention matrices alone. The
KV cache eliminates most of this redundancy.

---
## 2. Simple Cache: List-Based KV Storage

The simplest possible KV cache just appends tensors to a Python list. This is easy to understand
and works for small experiments. Let's explore `SimpleKVCache` from `cache.py`.

In [ ]:
# Create a simple list-based cache
simple_cache = SimpleKVCache()
print(f"Cache type: {type(simple_cache).__name__}")
print(f"Initial keys: {simple_cache.k}")
print(f"Initial values: {simple_cache.v}")

In [ ]:
# Simulate 5 generation steps: each step produces one k,v pair
print("Adding k,v tensors step by step:\n")

for step in range(5):
    k_new = torch.randn(1, 4, 1, 16)  # (batch, heads, 1 token, head_dim)
    v_new = torch.randn(1, 4, 1, 16)
    simple_cache.add(k_new, v_new)

    cached_k, cached_v = simple_cache.get()
    print(f"Step {step + 1}: cache holds {len(cached_k)} key tensors, "
          f"{len(cached_v)} value tensors")

# Retrieve and concatenate for attention
cached_k, cached_v = simple_cache.get()
k_all = torch.cat(cached_k, dim=2)  # Concatenate along sequence dimension
v_all = torch.cat(cached_v, dim=2)
print(f"\nConcatenated K shape: {k_all.shape}  (5 cached tokens)")
print(f"Concatenated V shape: {v_all.shape}")

The list-based approach is simple but has drawbacks: repeated `torch.cat` calls create new
tensors each time, causing memory allocation overhead. For production use, we pre-allocate
a fixed-size buffer and write into it.

---
## 3. Full KV Cache: Pre-Allocated Tensor Buffer

The `KVCache` class from `kv_cache.py` pre-allocates tensors of shape
`(batch_size, num_heads, max_seq_len, head_dim)` for both keys and values. New tokens are
written into the next available slot without any memory allocation.

In [ ]:
# Create a pre-allocated cache
cache = KVCache(
    batch_size=2,
    seq_len=64,
    num_heads=4,
    head_dim=16,
    device='cpu'
)

print(f"Cache created:")
print(f"  K buffer shape: {cache.k_cache.shape}")
print(f"  V buffer shape: {cache.v_cache.shape}")
print(f"  Max sequence length: {cache.max_seq_len}")
print(f"  Current length: {cache.cur_len}")
print(f"  Memory allocated: {cache.memory_usage():.4f} MB")

---
## 4. Simulating Autoregressive Generation Step by Step

Let's simulate token-by-token generation and watch the cache fill up. At each step:
1. We compute K, V for just the **new token** (shape `[batch, heads, 1, head_dim]`)
2. We `update` the cache with this single token's K, V
3. We `get` the full cached K, V for attention computation

In [ ]:
cache.reset()
print(f"{'Step':>4} | {'cur_len':>8} | {'Cached K shape':>25} | {'Cached V shape':>25}")
print("-" * 70)

for step in range(10):
    # Simulate computing k,v for one new token
    k_new = torch.randn(2, 4, 1, 16)
    v_new = torch.randn(2, 4, 1, 16)

    cache.update(k_new, v_new)
    k_cached, v_cached = cache.get()

    print(f"{step + 1:>4} | {cache.cur_len:>8} | {str(k_cached.shape):>25} | {str(v_cached.shape):>25}")

print(f"\nAfter 10 steps, cache holds {cache.cur_len} tokens.")
print(f"Underlying buffer is still shape {cache.k_cache.shape} (pre-allocated for 64 tokens).")

Notice that `get()` returns a **view** into the pre-allocated buffer (sliced to `cur_len`),
so there is no copy or allocation. The buffer shape stays fixed at `[2, 4, 64, 16]`
regardless of how many tokens have been cached so far.

---
## 5. Memory Usage Growth

Although the underlying buffer is fixed, let's examine how `memory_usage()` reports the total
allocated memory, and understand how it scales with sequence length.

In [ ]:
# Show how memory scales with max sequence length
seq_lengths_mem = [16, 32, 64, 128, 256, 512, 1024]
batch_size, num_heads, head_dim = 2, 4, 16

print(f"Memory usage for batch={batch_size}, heads={num_heads}, head_dim={head_dim}:\n")
print(f"{'Max Seq Len':>12} | {'Memory (MB)':>12}")
print("-" * 28)

memory_values = []
for sl in seq_lengths_mem:
    c = KVCache(batch_size, sl, num_heads, head_dim)
    mem = c.memory_usage()
    memory_values.append(mem)
    print(f"{sl:>12} | {mem:>11.4f}")

# Plot the growth
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(seq_lengths_mem, memory_values, 'o-', linewidth=2, color='steelblue')
ax.set_xlabel('Max Sequence Length')
ax.set_ylabel('Memory (MB)')
ax.set_title('KV Cache Memory vs Max Sequence Length')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nMemory grows linearly with sequence length (fixed batch, heads, head_dim).")

**Key observation**: KV cache memory is `2 * batch_size * num_heads * seq_len * head_dim * 4 bytes`.
The factor of 2 accounts for storing both keys and values. For real models with many layers,
multiply by `num_layers` as well -- each layer maintains its own KV cache.

---
## 6. Cached Attention in Action

The `attention_with_cache` function computes attention for a **single new query token** against
the full set of cached keys and values. This is the core operation during cached generation.

In [ ]:
# Reset and populate the cache with 8 tokens
cache.reset()
for _ in range(8):
    cache.update(
        torch.randn(2, 4, 1, 16),
        torch.randn(2, 4, 1, 16)
    )

k_cached, v_cached = cache.get()
print(f"Cached K shape: {k_cached.shape}  (8 tokens in cache)")
print(f"Cached V shape: {v_cached.shape}")

# New query: single token
q = torch.randn(2, 4, 1, 16)
scale = 16 ** 0.5

output, weights = attention_with_cache(q, k_cached, v_cached, scale)
print(f"\nQuery shape: {q.shape}  (1 new token)")
print(f"Output shape: {output.shape}  (1 output token)")
print(f"Attention weights shape: {weights.shape}  (1 query attending to 8 cached tokens)")

In [ ]:
# Visualize attention weights for one batch element, one head
fig, axes = plt.subplots(1, 4, figsize=(14, 3))

for head in range(4):
    w = weights[0, head, 0, :].detach().numpy()  # batch 0, head h, query 0
    axes[head].bar(range(len(w)), w, color='steelblue', alpha=0.8)
    axes[head].set_title(f'Head {head}')
    axes[head].set_xlabel('Cached Token Position')
    if head == 0:
        axes[head].set_ylabel('Attention Weight')
    axes[head].set_ylim(0, 1)

fig.suptitle('Attention Weights: New Token Attending to 8 Cached Tokens', y=1.02)
plt.tight_layout()
plt.show()

Each head learns different attention patterns. The new query token computes a softmax
distribution over all 8 cached positions, then uses those weights to produce a weighted
sum of the cached values. This is identical to standard attention -- the only difference
is that K and V come from the cache rather than being freshly computed.

---
## 7. Growing the Cache: Step-by-Step Generation

Let's simulate a full generation loop and visualize how attention patterns evolve as the
cache fills with more context.

In [ ]:
# Full generation simulation: 16 steps, watch attention evolve
gen_cache = KVCache(batch_size=1, seq_len=20, num_heads=4, head_dim=16)

all_weights = []  # Collect weights at each step

for step in range(16):
    # New token's k, v
    k_new = torch.randn(1, 4, 1, 16)
    v_new = torch.randn(1, 4, 1, 16)
    gen_cache.update(k_new, v_new)

    # Query for the next token
    q = torch.randn(1, 4, 1, 16)
    k_cached, v_cached = gen_cache.get()
    _, weights = attention_with_cache(q, k_cached, v_cached, scale=16 ** 0.5)
    all_weights.append(weights[0, 0, 0, :].detach().numpy())  # Head 0

# Plot attention evolution for head 0
fig, ax = plt.subplots(figsize=(10, 6))
max_len = max(len(w) for w in all_weights)
weight_matrix = np.zeros((len(all_weights), max_len))
for i, w in enumerate(all_weights):
    weight_matrix[i, :len(w)] = w

im = ax.imshow(weight_matrix, aspect='auto', cmap='Blues', interpolation='nearest')
ax.set_xlabel('Cached Token Position')
ax.set_ylabel('Generation Step')
ax.set_title('Attention Weight Evolution During Generation (Head 0)')
plt.colorbar(im, ax=ax, label='Attention Weight')
plt.tight_layout()
plt.show()

print("Each row shows the attention distribution at a given generation step.")
print("As more tokens enter the cache, attention spreads across more positions.")

The heatmap shows how each new generation step (y-axis) distributes attention across all
previously cached tokens (x-axis). Early steps have few cached tokens, so attention is
concentrated. Later steps spread attention across the full context.

---
## 8. Benchmarks: With vs Without KV Cache

Now let's run a proper benchmark comparing generation speed with and without the KV cache.
The `benchmark_with_without_cache` function simulates multi-step generation under both
strategies and measures wall-clock time.

In [ ]:
results = benchmark_with_without_cache(seq_lengths=[8, 16, 32, 64, 128])

In [ ]:
# Display results as a table
print(f"\n{'Seq Len':>8} | {'Without Cache (s)':>18} | {'With Cache (s)':>15} | {'Speedup':>8}")
print("-" * 56)
for i in range(len(results['seq_len'])):
    print(f"{results['seq_len'][i]:>8} | "
          f"{results['without_cache'][i]:>18.4f} | "
          f"{results['with_cache'][i]:>15.4f} | "
          f"{results['speedup'][i]:>7.2f}x")

In [ ]:
plot_cache_benchmarks(results)

**Analysis**: The left plot shows that generation time without a cache grows much faster than
with a cache. The right plot shows the speedup factor, which generally increases with sequence
length. Longer sequences benefit more from caching because they avoid recomputing more
historical K/V projections.

Note: these benchmarks use CPU timing. On GPU, the speedup would be even more pronounced due
to the higher overhead of redundant kernel launches without caching.

---
## 9. Memory Analysis: Cache Size Across Configurations

KV caching trades memory for speed. The `visualize_cache_memory` function shows how memory
requirements scale with batch size and sequence length.

In [ ]:
visualize_cache_memory()

Memory grows **linearly** with both batch size and sequence length. For large batch sizes
and long sequences, the KV cache can consume a significant fraction of available GPU memory.
This is one of the primary constraints on serving throughput.

---
## 10. Cache Efficiency Analysis

The `analyze_cache_efficiency` function examines how much computation the cache saves as a
function of sequence length, and plots the memory-vs-speed tradeoff.

In [ ]:
analyze_cache_efficiency()

**Left plot**: Cache efficiency (computation saved) approaches 100% rapidly. Even at sequence
length 10, the cache avoids recomputing 90% of K/V projections. By length 100, it avoids 99%.

**Right plot**: The memory-vs-speed tradeoff shows that KV cache memory is essentially fixed
(determined by max sequence length), while the speedup depends on how many tokens we generate.

---
## 11. Manual Analysis: Cache Memory vs Model Size

Real models vary in the number of attention heads and head dimension. Let's explore how
these hyperparameters affect KV cache memory requirements for a fixed sequence length.

In [ ]:
# Vary num_heads and head_dim, keeping batch=1, seq_len=512
head_configs = [
    (4, 16, "Tiny"),
    (8, 32, "Small"),
    (12, 64, "Medium (GPT-2)"),
    (16, 64, "Large"),
    (32, 128, "XL (LLaMA-7B)"),
    (40, 128, "XXL (LLaMA-13B)"),
]

seq_len_fixed = 512
batch_fixed = 1

print(f"KV cache memory for batch={batch_fixed}, seq_len={seq_len_fixed}:\n")
print(f"{'Config':>20} | {'Heads':>6} | {'Head Dim':>9} | {'d_model':>8} | {'Memory (MB)':>12}")
print("-" * 65)

config_names = []
config_memory = []

for n_heads, h_dim, name in head_configs:
    c = KVCache(batch_fixed, seq_len_fixed, n_heads, h_dim)
    mem = c.memory_usage()
    d_model = n_heads * h_dim
    print(f"{name:>20} | {n_heads:>6} | {h_dim:>9} | {d_model:>8} | {mem:>11.4f}")
    config_names.append(name)
    config_memory.append(mem)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(config_names, config_memory, color='steelblue', alpha=0.8)
ax.set_xlabel('KV Cache Memory (MB)')
ax.set_title(f'KV Cache Memory by Model Size (batch=1, seq_len={seq_len_fixed}, single layer)')

# Add value labels
for bar, mem in zip(bars, config_memory):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            f'{mem:.2f} MB', va='center', fontsize=9)

ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print("Note: Real models have 12-80+ layers. Multiply these numbers by num_layers")
print("for the total KV cache memory of the full model.")

In [ ]:
# Multi-layer scaling: show total cache for a full model
print("Total KV cache memory for full models (batch=1, seq_len=2048):\n")

model_configs = [
    ("GPT-2 Small", 12, 12, 64, 2048),
    ("GPT-2 Medium", 24, 16, 64, 2048),
    ("GPT-2 Large", 36, 20, 64, 2048),
    ("LLaMA-7B", 32, 32, 128, 2048),
]

print(f"{'Model':>16} | {'Layers':>7} | {'Heads':>6} | {'h_dim':>6} | {'Per Layer (MB)':>15} | {'Total (MB)':>11}")
print("-" * 72)

for name, n_layers, n_heads, h_dim, s_len in model_configs:
    per_layer = KVCache(1, s_len, n_heads, h_dim).memory_usage()
    total = per_layer * n_layers
    print(f"{name:>16} | {n_layers:>7} | {n_heads:>6} | {h_dim:>6} | {per_layer:>14.2f} | {total:>10.2f}")

For LLaMA-7B at sequence length 2048, the KV cache alone requires hundreds of MB per
request. When serving many concurrent users (large batch sizes), cache memory becomes the
primary bottleneck, not model weights.

---
## 12. Cache Reset and Reuse

Between generation requests, the cache can be `reset()` to zero cost -- no deallocation or
reallocation needed. The pre-allocated buffer is simply reused.

In [ ]:
# Demonstrate reset
demo_cache = KVCache(batch_size=1, seq_len=32, num_heads=4, head_dim=16)

# Fill with some data
for _ in range(10):
    demo_cache.update(torch.randn(1, 4, 1, 16), torch.randn(1, 4, 1, 16))
print(f"Before reset: cur_len = {demo_cache.cur_len}")

demo_cache.reset()
print(f"After reset:  cur_len = {demo_cache.cur_len}")
print(f"Buffer still allocated: {demo_cache.k_cache.shape}")
print(f"Memory still reserved:  {demo_cache.memory_usage():.4f} MB")
print("\nReset is O(1) -- it just moves the write pointer back to 0.")

---
## 13. Verifying Correctness: Cached vs Non-Cached Attention

An important sanity check: cached attention must produce **identical** results to full
recomputation. Let's verify this numerically.

In [ ]:
torch.manual_seed(123)
batch, heads, head_d = 1, 4, 16
n_tokens = 8

# Generate all K, V upfront
all_k = torch.randn(batch, heads, n_tokens, head_d)
all_v = torch.randn(batch, heads, n_tokens, head_d)
q_last = torch.randn(batch, heads, 1, head_d)

# Method 1: Full attention (non-cached) -- only care about last query row
q_full = torch.cat([torch.randn(batch, heads, n_tokens - 1, head_d), q_last], dim=2)
scale = head_d ** 0.5
scores_full = torch.matmul(q_full, all_k.transpose(-2, -1)) / scale

# Causal mask
mask = torch.triu(torch.ones(n_tokens, n_tokens), diagonal=1).bool()
scores_full.masked_fill_(mask.unsqueeze(0).unsqueeze(0), float('-inf'))
weights_full = torch.softmax(scores_full, dim=-1)
out_full = torch.matmul(weights_full, all_v)
out_full_last = out_full[:, :, -1:, :]  # Last token's output

# Method 2: Cached attention
out_cached, _ = attention_with_cache(q_last, all_k, all_v, scale)

# Compare
diff = (out_full_last - out_cached).abs().max().item()
print(f"Max absolute difference: {diff:.2e}")
print(f"Results match: {diff < 1e-5}")
print("\nCached attention produces identical outputs to full recomputation")
print("(for the last token's query against all previous keys/values).")

The tiny numerical difference (if any) is due to floating-point precision, not a correctness
issue. The cached approach computes mathematically identical attention for the new token.

---
## Key Insights

**Why KV Cache Exists**
- Autoregressive generation produces one token at a time. Without caching, each new token
  requires recomputing K and V for the entire history, leading to O(n^2) total work over n steps.
- The KV cache stores previously computed K and V, so each step only computes for the new token.

**Speed Benefits**
- Speedup increases with sequence length. For a 128-token sequence, caching can provide
  significant wall-clock improvements.
- The cached approach reduces per-step K/V computation from O(n) to O(1), though attention
  itself is still O(n) per step (querying against n cached entries).

**Memory Tradeoffs**
- KV cache memory = `2 * num_layers * batch_size * num_heads * seq_len * head_dim * 4 bytes`.
- For large models (e.g., LLaMA-7B with 32 layers), the cache for a single 2048-token
  sequence can require hundreds of MB.
- At serving time with many concurrent users, KV cache memory often exceeds model weight
  memory and becomes the primary constraint on throughput.

**Implementation Choices**
- **List-based cache** (SimpleKVCache): Easy to implement but requires `torch.cat` at each
  step, allocating new tensors. Suitable for prototyping.
- **Pre-allocated cache** (KVCache): Fixed memory allocation, zero-copy reads via slicing.
  Used in production systems for predictable memory behavior.

**Advanced Techniques** (beyond this notebook)
- **Grouped-Query Attention (GQA)**: Shares K/V across query groups, reducing cache size.
- **Sliding Window Attention**: Evicts old cache entries, bounding memory at the cost of
  limited context.
- **Quantized KV Cache**: Stores cached K/V in lower precision (e.g., FP8) to halve memory.
- **PagedAttention** (vLLM): Manages cache memory in pages for efficient multi-request serving.